做z-score normalization>>mean=0,std=1

In [6]:
from monai.transforms import NormalizeIntensity

normalize = NormalizeIntensity()

normalized_img = normalize(img)

print(normalized_img.mean())
print(normalized_img.std())

metatensor(-1.3115e-07)
metatensor(1.)


建立測試資料

In [1]:
import os

from nilearn import datasets

data = datasets.fetch_development_fmri(n_subjects=1)

mri_path = data.func[0]

print(mri_path)

[fetch_development_fmri] Dataset directory found: /Users/liaoyubo/nilearn_data/development_fmri
[fetch_development_fmri] Dataset directory found: /Users/liaoyubo/nilearn_data/development_fmri/development_fmri
[fetch_development_fmri] Dataset directory found: /Users/liaoyubo/nilearn_data/development_fmri/development_fmri
/Users/liaoyubo/nilearn_data/development_fmri/development_fmri/sub-pixar123_task-pixar_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz


使用MONAI LoadImage
numpy.ndarray>>MetaTensor
AI 模型使用PyTorch Tensor
NIfTI>>MONAI>>Tensor>>Deep learning


In [2]:
from monai.transforms import LoadImage


loader = LoadImage(image_only=True)

img = loader(mri_path)

print(type(img))
print(img.shape)

<class 'monai.data.meta_tensor.MetaTensor'>
torch.Size([50, 59, 50, 168])


tensor和metetensor的差異
tensor只有數值
metetensor有影像數值＋影像空間資訊

dim[0]=4                    #4d image
affine: tensor              #voxel index轉換成真實世界座標
[   4.,    0.,    0.,  -96.],
[   0.,    4.,    0., -132.],
[   0.,    0.,    4.,  -78.],

space: RAS                  ＃醫學影像分析需要統一方向
代表
R = Right
A = Anterior
S = Superior

In [3]:
print(img.meta)


{'sizeof_hdr': array(348, dtype=int32), 'extents': array(0, dtype=int32), 'session_error': array(0, dtype=int16), 'dim_info': array(0, dtype=uint8), 'dim': array([  4,  50,  59,  50, 168,   1,   1,   1], dtype=int16), 'intent_p1': array(0., dtype=float32), 'intent_p2': array(0., dtype=float32), 'intent_p3': array(0., dtype=float32), 'intent_code': array(0, dtype=int16), 'datatype': array(256, dtype=int16), 'bitpix': array(8, dtype=int16), 'slice_start': array(0, dtype=int16), 'pixdim': array([  1.     ,   4.     ,   4.     ,   4.     , 180.90053,   1.     ,
         1.     ,   1.     ], dtype=float32), 'vox_offset': array(0., dtype=float32), 'scl_slope': array(nan, dtype=float32), 'scl_inter': array(nan, dtype=float32), 'slice_end': array(0, dtype=int16), 'slice_code': array(0, dtype=uint8), 'xyzt_units': array(0, dtype=uint8), 'cal_max': array(0., dtype=float32), 'cal_min': array(0., dtype=float32), 'slice_duration': array(0., dtype=float32), 'toffset': array(0., dtype=float32), 'glma

開始MONAI Preprocessing pipeline
metatensor>>model input
流程：
MRI
 ↓
EnsureChannelFirst
 ↓
Normalize intensity
 ↓
Tensor
 ↓
AI model

In [4]:
from monai.transforms import (
    Compose,
    EnsureChannelFirst,
    ScaleIntensity
)


transform = Compose([
    EnsureChannelFirst(),
    ScaleIntensity()
])


processed_img = transform(img)

print(processed_img.shape)
print(processed_img.min())
print(processed_img.max())

torch.Size([168, 50, 59, 50])
metatensor(0.)
metatensor(1.)


在ai中通常第一維通常代表channel,但在fMRI中time被誤認為channel (168是time)


In [5]:
print(img.shape)
print(processed_img.shape)
print(img.meta["original_channel_dim"])


torch.Size([50, 59, 50, 168])
torch.Size([168, 50, 59, 50])
-1
